# 🥇 CME Gold Futures (GC) Market Microstructure & Quantitative Alpha
### **Lead Researcher:** Ayush Pandey (AP) &nbsp;|&nbsp; **Collaborator:** Bhaskar (`bhaskarauthor@gmail.com`)
### **Asset:** COMEX Gold Futures (`GC`) & CME Globex MDP 3.0 / Physical Gold Equiv (`PAXG/USD`)

---

## 📌 Research Overview
This notebook provides a complete quantitative exploration of high-frequency market microstructure for CME Gold Futures:
1. **Level 1 Order Book Imbalance (OBI):** Measuring buying vs selling queue pressure at the Best Bid & Offer (BBO).
2. **Volume-Weighted Micro-Price:** Evaluating the fair-value price adjusted for order book depth.
3. **Predictive Alpha Decay:** Information Coefficient (IC) analysis across $k \in \{1, 3, 5, 10, 20\}$ tick horizons.
4. **Cross-Asset Arbitrage:** Gold/Silver Ratio (GSR) mean-reversion and Micro-Gold basis tracking.

In [ ]:
import os
import sys
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# Set paths dynamically whether running in notebooks/ or repository root
BASE_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
DB_PATH = DATA_DIR / "market_store.db"

print(f"[✓] Base directory: {BASE_DIR.resolve()}")
print(f"[✓] Data directory: {DATA_DIR.resolve()}")
print(f"[✓] Database exists: {DB_PATH.exists()}")

## 📊 1. Load Intraday 1-Minute CME Gold Futures (`GC=F`)
We query the local SQLite Market Store to load 7,600+ bars of continuous 1-minute gold futures data.

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
query = """
SELECT ts_event, symbol, open, high, low, close, volume, vwap, est_spread
FROM intraday_bars
ORDER BY ts_event ASC
"""
df_bars = pd.read_sql_query(query, conn)
df_bars["ts_event"] = pd.to_datetime(df_bars["ts_event"])

print(f"Total Bars: {len(df_bars):,}")
print(f"Date Range: {df_bars['ts_event'].min()} -> {df_bars['ts_event'].max()}")
print(f"Latest Close: ${df_bars['close'].iloc[-1]:.2f}")
print(f"Avg Spread:   ${df_bars['est_spread'].mean():.4f}")
df_bars.tail(5)

## 🔬 2. Level 1 Order Book Dynamics (MBP-1)
We load MBP-1 tick and depth events to compute:
- **Normalized Order Book Imbalance (OBI):**
  $$\text{OBI}_t = \frac{Q_t^b - Q_t^a}{Q_t^b + Q_t^a} \in [-1, +1]$$
- **Volume-Weighted Micro-Price:**
  $$P_t^{\text{micro}} = \frac{Q_t^b P_t^a + Q_t^a P_t^b}{Q_t^b + Q_t^a}$$

In [ ]:
query_ob = """
SELECT ts_recv, symbol, action, side, price, size, bid_px_00, ask_px_00, bid_sz_00, ask_sz_00, spread, obi, micro_price
FROM orderbook_events
ORDER BY ts_recv ASC
"""
df_ob = pd.read_sql_query(query_ob, conn)
print(f"Total Order Book Events: {len(df_ob):,}")
print(f"Average OBI:             {df_ob['obi'].mean():+.4f}")
print(f"Average Spread:          ${df_ob['spread'].mean():.4f}")
df_ob.head(5)

## 📈 3. Multi-Horizon Information Coefficient (IC) Analysis
We test if Order Book Imbalance and Micro-Price deviations predict mid-price returns over future horizons $k \in \{1, 3, 5, 10, 20\}$ ticks.

In [ ]:
df_ob["mid_price"] = (df_ob["bid_px_00"] + df_ob["ask_px_00"]) / 2.0
df_ob["micro_spread"] = df_ob["micro_price"] - df_ob["mid_price"]

horizons = [1, 3, 5, 10, 20]
ic_data = []

for k in horizons:
    df_ob[f"fwd_ret_{k}"] = (df_ob["mid_price"].shift(-k) - df_ob["mid_price"]) / df_ob["mid_price"]
    valid = df_ob[["obi", "micro_spread", f"fwd_ret_{k}"]].dropna()
    
    p_obi = valid["obi"].corr(valid[f"fwd_ret_{k}"])
    p_micro = valid["micro_spread"].corr(valid[f"fwd_ret_{k}"])
    s_micro = valid["micro_spread"].rank().corr(valid[f"fwd_ret_{k}"].rank())
    
    ic_data.append({
        "Horizon (k Ticks)": f"k = {k}",
        "Micro-Price Pearson IC": round(p_micro, 4),
        "Micro-Price Spearman Rank IC": round(s_micro, 4),
    })

pd.DataFrame(ic_data)

## ⚖️ 4. Cross-Asset Metals Arbitrage & Gold/Silver Ratio (GSR)
We evaluate the macroeconomic Gold/Silver Ratio and check for statistical arbitrage z-score dislocations.

In [ ]:
from scripts.mine_metals_cross_asset import MetalsMiner

miner = MetalsMiner()
quotes = miner.fetch_quotes()

p_gold = quotes["GC=F"]["regularMarketPrice"]
p_silver = quotes["SI=F"]["regularMarketPrice"]
p_mgc = quotes["MGC=F"]["regularMarketPrice"]

gsr = p_gold / p_silver
basis_dislocation = p_mgc - p_gold

print(f"Gold (GC=F):           ${p_gold:,.2f}")
print(f"Silver (SI=F):         ${p_silver:,.2f}")
print(f"Gold/Silver Ratio:     {gsr:.2f}")
print(f"Micro Gold Basis (bps): {(basis_dislocation / p_gold) * 10000:.2f} bps")